# Qwen3-0.6B × stock vLLM FlashInfer backend (reference)

이 노트북은 **vLLM 내장 `AttentionBackendEnum.FLASHINFER`** 만 사용하는 reference 다.
옆 자리의 `qwen3_flashinfer_attention.ipynb` (커스텀 플러그인 = `MyFlashInferBackend`)
가 에러를 낼 때, **stock backend 자체는 정상으로 도는가** 를 먼저 확인하기 위한 베이스라인.

차이 요약:

| | reference (이 노트북) | educational (옆 노트북) |
|---|---|---|
| backend slot | `FLASHINFER` (vLLM 빌트인) | `CUSTOM` (entry point 로 register) |
| 구현체 | `vllm.v1.attention.backends.flashinfer.FlashInferBackend` | `flashinfer_attention_backend:MyFlashInferBackend` |
| plugin 의존 | 없음 (`pip install -e .` 불필요) | 있음 |
| 관찰 로그 | 없음 (stock 은 educational 로그 없음) | `MyFlashInferImpl.forward fired ...` |

**용도**: 환경 (vllm + flashinfer + Qwen3-0.6B) 의 동작 여부를 분리 검증.
여기서 PASS 하면 → 문제는 **커스텀 backend 코드** 안에 있다고 좁힐 수 있다.
여기서 FAIL 하면 → 환경 (FlashInfer 휠 / vLLM 버전 / GPU) 자체가 문제.

## 1. 환경 확인

vLLM 0.19.x + flashinfer-python >= 0.2.0 + CUDA GPU.

In [1]:
import torch, vllm
import flashinfer

print('cuda      :', torch.cuda.is_available())
print('gpu       :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm      :', vllm.__version__)
print('flashinfer:', getattr(flashinfer, '__version__', 'unknown'))

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 stock FlashInferBackend 시그니처가 다를 수 있음.'
)

cuda      : True
gpu       : NVIDIA GeForce RTX 5090
vllm      : 0.19.1
flashinfer: 0.6.6


## 2. Qwen3-0.6B 구조 확인

FlashInfer 제약: `head_dim ∈ {64, 128, 256}`, `block_size ∈ {1, 16, 32, 64}`.
Qwen3-0.6B 는 `head_dim=128`, vLLM 기본 `block_size=16` 이므로 둘 다 OK.

In [2]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden  :', cfg.hidden_size)
print('Q heads :', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers  :', cfg.num_hidden_layers)

assert hd in (64, 128, 256), 'FlashInfer 는 head_dim ∈ {64,128,256} 만 허용'

/home/osehn/orchestrate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


hidden  : 1024
Q heads : 16 / KV heads: 8
head_dim: 128
layers  : 28


## 3. Backend slot 확인

`AttentionBackendEnum.FLASHINFER` 가 vLLM 빌트인 `FlashInferBackend` 클래스 경로로 매핑되어
있는지 확인. 이 노트북은 **CUSTOM 슬롯을 건드리지 않는다** — 따라서 옆 노트북의 플러그인이
함께 등록되어 있어도 무관.

In [3]:
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.FLASHINFER.get_path()
print('FLASHINFER slot ->', path)
assert path.endswith('FlashInferBackend'), f'예상 외 경로: {path}'

FLASHINFER slot -> vllm.v1.attention.backends.flashinfer.FlashInferBackend


## 4. LLM 로드

`attention_backend=AttentionBackendEnum.FLASHINFER` 한 줄로 stock FlashInfer 가 붙는다.
옆 노트북과 동일한 (`max_num_batched_tokens=64`) 설정으로 chunked prefill 도 트리거되도록
두지만, 이 노트북은 동작 검증만이 목적이라 실행 증거 로그 관찰은 하지 않는다.

In [4]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.FLASHINFER,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
    max_num_batched_tokens=64,
)

INFO 05-07 22:28:43 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'max_num_batched_tokens': 64, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'attention_backend': <AttentionBackendEnum.FLASHINFER: 'vllm.v1.attention.backends.flashinfer.FlashInferBackend'>}
INFO 05-07 22:28:45 [model.py:549] Resolved architecture: Qwen3ForCausalLM
WARNING 05-07 22:28:45 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-07 22:28:45 [model.py:1678] Using max model len 2048
INFO 05-07 22:28:45 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=64.
INFO 05-07 22:28:45 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-07 22:28:45 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-07 22:28:45 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during i

(EngineCore pid=2591257) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=2591257) INFO 05-07 22:28:52 [weight_utils.py:625] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.48it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.48it/s]
(EngineCore pid=2591257) 


(EngineCore pid=2591257) INFO 05-07 22:28:53 [default_loader.py:384] Loading weights took 0.68 seconds
(EngineCore pid=2591257) INFO 05-07 22:28:53 [gpu_model_runner.py:4820] Model loading took 1.12 GiB memory and 1.974031 seconds
(EngineCore pid=2591257) INFO 05-07 22:28:54 [gpu_worker.py:436] Available KV cache memory: 26.9 GiB
(EngineCore pid=2591257) INFO 05-07 22:28:54 [kv_cache_utils.py:1319] GPU KV cache size: 251,808 tokens
(EngineCore pid=2591257) INFO 05-07 22:28:54 [kv_cache_utils.py:1324] Maximum concurrency for 2,048 tokens per request: 122.95x
(EngineCore pid=2591257) INFO 05-07 22:28:54 [kernel_warmup.py:69] Warming up FlashInfer attention.


(EngineCore pid=2591257) 2026-05-07 22:28:54,297 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2591257) 2026-05-07 22:28:54,316 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=2591257) INFO 05-07 22:28:54 [core.py:283] init engine (profile, create kv cache, warmup model) took 0.90 seconds
(EngineCore pid=2591257) INFO 05-07 22:28:56 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=2591257) WARNING 05-07 22:28:56 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=2591257) WARNING 05-07 22:28:56 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=2591257) INFO 05-07 22:28:56 [vllm.py:1025] Cudagraph is disabled under eager mode
(EngineCore pid=2591257) INFO 05-07 22:28:56 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant


## 5. Generate

옆 노트북과 동일한 prompt 셋. 긴 prompt 가 chunked prefill 로 쪼개진다.

In [5]:
long_prompt = (
    'In the long history of artificial intelligence research, from the early '
    'symbolic AI of the 1950s through the neural network revival of the 1980s, '
    'the deep learning breakthroughs of the 2010s, and the transformer-based '
    'large language models of the 2020s, one theme has remained constant: '
    'the answer is'
)

prompts = [
    'The capital of France is',
    long_prompt,
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}] (prompt {len(prompts[i])} chars) {o.outputs[0].text[:80]}')

Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 29.60it/s, est. speed input: 665.85 toks/s, output: 478.76 toks/s]

[0] (prompt 24 chars)  Paris. The capital of Italy is Rome. The capital of Spain is Madrid.
[1] (prompt 300 chars)  the key to solving the problem. This is the core of the problem. The
[2] (prompt 26 chars)  "Macbeth" in 1606. The play is based on
[3] (prompt 21 chars)  a group of developers who wanted to create a new way to write code that is


## 6. PASS 기준

- 위 셀이 traceback 없이 끝나고
- 4개 prompt 모두 비어 있지 않은 텍스트가 출력되면

→ stock FlashInfer backend × Qwen3-0.6B × 이 환경 조합은 **건강하다**. 옆 노트북
(`qwen3_flashinfer_attention.ipynb`) 의 에러는 커스텀 `MyFlashInferBackend` 코드
안쪽에서 발생한 것으로 좁혀진다 — `flashinfer_attention_backend.py` 의 Metadata /
Builder / Impl 셋 중 하나.

반대로 이 셀이 실패하면 — 환경 자체가 문제. 점검 순서:
1. `flashinfer.__version__` 가 0.2 이상인가
2. `vllm.__version__` 가 0.19.x 인가
3. CUDA / driver 버전이 flashinfer 휠과 호환되는가 (`flashinfer-python` 의 wheel
   tags 와 `nvidia-smi` 의 driver 비교)